# SG Property, SPY, and US 10Y Report

Reproduces the local HTML report and documents the data alignment choices. Outputs are intentionally cleared for version control.


In [ ]:
from pathlib import Path

import pandas as pd
import yfinance as yf


In [ ]:
PROJECT = Path.cwd().parents[1] if Path.cwd().name == '30_exposure' else Path.cwd()
DATA_PATH = PROJECT / 'data/interim/sg_private_residential_property_price_index_quarterly.csv'
REPORT_PATH = PROJECT / 'reports/sg_property_spy_10y_report.html'
SG_COLS = ['All Residential', 'Landed', 'Non-Landed']


## Load Data

The saved SG data is cleaned into tidy quarterly rows. Market data is pulled from Yahoo Finance. SPY is backfilled with ^GSPC before SPY inception by scaling ^GSPC to SPY at the first available month/quarter-end.


In [ ]:
sg = pd.read_csv(DATA_PATH, parse_dates=['quarter_end'])
sg_wide = sg.pivot(index='quarter_end', columns='property_type', values='index').sort_index()
raw = yf.download(['SPY', '^GSPC', '^TNX'], start='1989-01-01', auto_adjust=True, progress=False, group_by='column')
close = raw['Close']
market_daily = pd.DataFrame({'SPY': close['SPY'], 'GSPC': close['^GSPC'], 'US10Y': close['^TNX']}).dropna(how='all')


## Analysis Notes

Use month-end points for the levels chart. Use quarter-end points for correlations and the 3x property-return thought experiment, because SG property returns are quarterly observations.


In [ ]:
# See the generated report for the full plotting code. The key 3x series is:
base_q = pd.Timestamp('1990-03-31')
q_dates = pd.date_range(base_q, sg_wide.index.max(), freq='QE')
sg_q = sg_wide[SG_COLS].reindex(sg_wide.index.union(q_dates)).sort_index().ffill().loc[q_dates]
all_res_3x = (1 + sg_q['All Residential'].pct_change().mul(3.0)).cumprod().mul(100.0)
all_res_3x.loc[base_q] = 100.0
all_res_3x = all_res_3x.sort_index()
all_res_3x.tail()


## Report

The latest generated HTML is `reports/sg_property_spy_10y_report.html`.
